In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path

root = Path('.')
df = pd.read_csv(root/'events_scored.csv')
len(df), df.head(2)

## Choose a threshold
We use the 3rd percentile of scores (lower is more anomalous). You can tune this later based on precision/recall.

In [ ]:
threshold = float(np.quantile(df['iso_score'].values, 0.03))
df['iso_anomaly'] = df['iso_score'] < threshold
threshold, df['iso_anomaly'].mean()

## Split clean vs quarantine and save outputs

In [ ]:
clean = df.loc[~df['iso_anomaly']].copy()
quarantine = df.loc[df['iso_anomaly']].copy()
clean.to_csv(root/'events_clean.csv', index=False)
quarantine.to_csv(root/'events_quarantine.csv', index=False)
print('Saved clean/quarantine datasets')
len(clean), len(quarantine)

## Quick evaluation (using our injected labels)
In a real system we don't know the ground truth. Here we use the `anomaly_type` column (added when we generated data) to estimate precision/recall.

In [ ]:
y_true = df['anomaly_type'].ne('base')  # True if injected
y_pred = df['iso_anomaly']
tp = int(((y_true==1)&(y_pred==1)).sum())
fp = int(((y_true==0)&(y_pred==1)).sum())
fn = int(((y_true==1)&(y_pred==0)).sum())
precision = tp / max(1, tp+fp)
recall = tp / max(1, tp+fn)
f1 = 2*precision*recall/max(1e-9, precision+recall)
metrics = {'precision': float(precision), 'recall': float(recall), 'f1': float(f1), 'threshold': threshold}
with open(root/'report_metrics.json','w') as f: json.dump(metrics,f,indent=2)
metrics